# Pontos no mapa: GeoPandas e folium com registros do Fogo Cruzado

Notebook da aula **Pontos no mapa**: GeoPandas e folium com registros do Fogo
Cruzado para a cidade do Rio de Janeiro. Há uma parte para cada exercício da
aula, na mesma ordem:

1. aquecimento com pandas
2. GeoDataFrame
3. junção espacial
4. contagens e taxas
5. formas
6. o mapa
7. (extra) comece o seu trabalho

**Como rodar.** No Google Colab, pandas, geopandas, shapely, folium e matplotlib
já estão instalados: é só rodar as células (a primeira instala só o que faltar).
Localmente, `pip install geopandas folium babel` basta.

**Os dados.**

- **Registros do Fogo Cruzado**, `fogo_cruzado_rio.csv`: uma linha por registro na
  base do Instituto Fogo Cruzado para a cidade do Rio de Janeiro, de 05/07/2016 a
  24/11/2025 (27.738 registros, 16 colunas). É uma base preparada para esta aula:
  use na aula, não redistribua. A célula de preparação carrega o arquivo de
  `DATA_URL` (o link que o professor passa; um link comum de compartilhamento do
  Google Drive funciona), de `data/` se o arquivo já estiver lá, ou pede o upload
  (Colab). Crédito sugerido: "Instituto Fogo Cruzado (api.fogocruzado.org.br).
  Extração preparada para a aula; campos simplificados."
- **Bairros e população**, do DATA.RIO, o portal de dados abertos da Prefeitura da
  Cidade do Rio de Janeiro (Instituto Pereira Passos, IPP), sob
  [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/):
  [Limite de Bairros](https://www.data.rio/datasets/PCRJ::limite-de-bairros/about)
  (167 polígonos, simplificados para cerca de 5 m) e Censo 2022, população por
  bairros (preliminar). A célula de preparação baixa os dois uma vez para
  `data/rio/` e reaproveita depois: do repositório público da aula, para que seus
  números batam com os slides, ou do DATA.RIO se o GitHub não responder.

**A janela da aula.** As aulas usam `WINDOW_START = "2023-01-01"` até o fim da
base, 24/11/2025: 5.129 registros. Troque a constante na célula de preparação e
tudo o que vem depois acompanha. A base inteira só é usada onde a célula disser.

**O que esta tabela não diz.**

- Uma linha é um *registro*: o Fogo Cruzado anota o que ficou sabendo, então
  registro não é evento.
- A tabela não tem denominador (nem habitantes, nem ruas), então contagem não é
  risco.
- `main_reason` é "Não identificado" em 42% dos registros da janela (61% da base) e
  67% dos registros da janela não têm vítimas (78% da base).
- Um registro (Penha, 28/10/2025) tem 145 vítimas; o segundo maior da janela tem 15.
- A coluna `neighborhood` é o rótulo do próprio Fogo Cruzado, não o polígono onde o
  ponto cai.

A pergunta por trás de toda parte é *comparado a quê?*

**Seus próprios pontos.** Tudo o que precisa só de coordenadas funciona com
qualquer CSV que tenha as colunas `latitude` e `longitude`: leia o seu arquivo no
lugar do arquivo do Fogo Cruzado. As células que usam as outras colunas
(`police_action`, `main_reason`, `neighborhood`, `n_victims`) são as únicas que
você precisaria adaptar.

As células marcadas com **FAÇA** são suas: escreva o código onde a célula diz `# seu código aqui`. As soluções ficam escondidas logo abaixo de cada exercício, na célula seguinte; "rodar tudo" funciona e define o que as células seguintes precisam.

## Preparação

In [ ]:
# O Colab já tem pandas, geopandas, shapely, folium e matplotlib: esta célula instala
# só o que faltar (o babel formata os números em pt-BR). No Colab, em geral não instala nada.
import importlib.util
import subprocess
import sys

NEEDED = ["pandas", "geopandas", "shapely", "folium", "matplotlib", "babel"]
missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("instalado:", ", ".join(missing))
else:
    print("nada a instalar")

In [ ]:
import pandas as pd
import geopandas as gpd
import shapely
import folium
from folium import plugins

In [ ]:
# O código abaixo precisa do geopandas 1.0 ou mais novo (GeoSeries.union_all) e do shapely 2.
import geopandas
import shapely

major, minor = (int(part) for part in geopandas.__version__.split(".")[:2])
assert (major, minor) >= (1, 0), (
    f"geopandas {geopandas.__version__} é antigo demais: rode  pip install -U geopandas")
assert int(shapely.__version__.split(".")[0]) >= 2, (
    f"shapely {shapely.__version__} é antigo demais: rode  pip install -U shapely")
print("geopandas", geopandas.__version__, "| shapely", shapely.__version__)

In [ ]:
WINDOW_START = "2023-01-01"  # a janela vai até 24/11/2025, o fim da base

In [ ]:
import unicodedata


def norm(name):
    """'Imperial de São Cristóvão' -> 'IMPERIAL DE SAO CRISTOVAO'."""
    text = unicodedata.normalize("NFKD", str(name))
    return text.encode("ascii", "ignore").decode().upper()

In [ ]:
from babel.numbers import format_decimal


def pt(valor, casas=0):
    """1030 -> '1.030'; 10.1 (casas=1) -> '10,1'. Babel, locale pt_BR."""
    padrao = "#,##0" + ("." + "0" * casas if casas else "")
    return format_decimal(valor, format=padrao, locale="pt_BR")

In [ ]:
# A base da aula: fogo_cruzado_rio.csv (27.738 registros, 16 colunas).
# Fonte: Instituto Fogo Cruzado (api.fogocruzado.org.br); extração preparada para a aula.
# É um arquivo da aula, não um conjunto aberto: use na aula, não redistribua.
import re
import urllib.request
from pathlib import Path

# >>> COLE AQUI o link que o professor passa, entre as aspas. <<<
# Um link comum de compartilhamento do Google Drive funciona ("Qualquer pessoa com o link" tem de estar ligado).
DATA_URL = "COLE-AQUI-O-LINK-DO-fogo_cruzado_rio.csv"


def direct_link(url):
    """Um link de compartilhamento do Google Drive vira o link de download direto; qualquer outra URL fica como está."""
    m = re.search(r"drive\.google\.com/.*?(?:/d/|[?&]id=)([\w-]+)", url)
    return ("https://drive.usercontent.google.com/download?id=" + m.group(1)
            + "&export=download") if m else url


FC_PATH = Path("data/fogo_cruzado_rio.csv")
FC_PATH.parent.mkdir(parents=True, exist_ok=True)

# 1. Já está aqui (de uma execução anterior, ou você copiou para data/)? Use.
# 2. Senão, baixe de DATA_URL, depois de preenchida.
if not FC_PATH.exists() and direct_link(DATA_URL).startswith("http"):
    try:
        urllib.request.urlretrieve(direct_link(DATA_URL), FC_PATH)
        if FC_PATH.read_bytes()[:1] == b"<":     # uma página da web, não o CSV (o link não é público?)
            FC_PATH.unlink()
            print("DATA_URL devolveu uma página da web, não o arquivo: o link está aberto a qualquer pessoa?")
        else:
            print("baixado", FC_PATH)
    except Exception as error:
        print("não foi possível baixar de DATA_URL:", error)

# 3. Último recurso: escolha o arquivo no seu computador (só no Colab).
if not FC_PATH.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"fogo_cruzado_rio.csv não está em {FC_PATH.resolve()}: coloque-o lá, "
            "ou cole o link em DATA_URL acima") from None
    print("Escolha fogo_cruzado_rio.csv no seu computador:")
    uploaded = files.upload()
    Path(next(iter(uploaded))).replace(FC_PATH)

# Uma checagem rápida de que este é o arquivo da aula.
EXPECTED = ["id", "date", "year", "latitude", "longitude", "neighborhood", "locality",
            "police_action", "agent_presence", "main_reason", "massacre", "n_victims",
            "n_dead", "n_wounded", "n_agents", "n_civilians"]
columns = list(pd.read_csv(FC_PATH, nrows=0).columns)
assert columns == EXPECTED, f"não é o arquivo da aula: as colunas são {columns}"
print("usando", FC_PATH, "|", sum(1 for _ in open(FC_PATH, encoding="utf-8")) - 1, "registros")

In [ ]:
# Os dois arquivos do IPP, guardados em data/rio/: os polígonos dos bairros e a população do Censo 2022.
# Fonte: Prefeitura da Cidade do Rio de Janeiro / Instituto Pereira Passos (IPP),
# DATA.RIO (https://www.data.rio), licença CC BY 4.0.
import json
import urllib.parse
import urllib.request
from pathlib import Path

DATA = Path("data/rio")
FILES = ["populacao_bairros.csv", "bairros.geojson"]
# Os mesmos dois arquivos dos slides (polígonos simplificados para cerca de 5 m), guardados no
# repositório público da aula num commit fixo, para que seus números batam com os slides.
CLASS_REPO = ("https://raw.githubusercontent.com/Luna-v0/Presenting/"
              "e2cb8b397136c10c47ada1d37718d4dd815f7859/"
              "manim-videos/decks/geo-folium/assets/data/")
# Se o GitHub não responder: o servidor ArcGIS da própria prefeitura, onde o DATA.RIO os guarda.
ARCGIS = "https://pgeo3.rio.rj.gov.br/arcgis/rest/services/"


def arcgis(layer, **params):
    """Consulta o servidor ArcGIS da prefeitura: um dict GeoJSON, ou uma tabela de atributos."""
    params = {"where": "1=1", "f": "json", **params}
    rows = []
    for _ in range(20):                         # a maior camada cabe numa página
        query = urllib.parse.urlencode({**params, "resultOffset": len(rows)})
        with urllib.request.urlopen(f"{ARCGIS}{layer}/query?{query}", timeout=120) as r:
            page = json.load(r)
        if params["f"] != "json":
            return page
        rows += [feature["attributes"] for feature in page["features"]]
        if not page.get("exceededTransferLimit"):
            return pd.DataFrame(rows)
    raise RuntimeError(f"{layer}: o servidor não parou de paginar")


def download_from_datario():
    pop = arcgis("Censo/Limites_administrativos_Censo_2022/MapServer/2",
                 returnGeometry="false", outFields="codbairro,nome,Total_de_pessoas_2022")
    pop = pd.DataFrame({"codbairro": pop["codbairro"].astype(int), "nome": pop["nome"],
                        "populacao_2022": pop["Total_de_pessoas_2022"]})

    geo = arcgis("Cartografia/Limites_administrativos/FeatureServer/4", f="geojson",
                 outSR=4326, outFields="codbairro,nome,regiao_adm,rp")
    bairros = gpd.GeoDataFrame.from_features(geo["features"], crs=4326)
    bairros["codbairro"] = bairros["codbairro"].astype(int)
    bairros["regiao_adm"] = bairros["regiao_adm"].str.strip()
    geoms = bairros.geometry.to_numpy()
    if hasattr(shapely, "coverage_simplify"):   # shapely >= 2.1: de ~4 MB para ~1 MB
        geoms = shapely.coverage_simplify(geoms, tolerance=0.00005)
    else:
        print("ATENÇÃO: o shapely < 2.1 não simplifica os polígonos como os slides "
              "(pip install -U shapely): a contagem de pontos na divisa pode diferir "
              "um pouco dos slides.")
    bairros["geometry"] = shapely.set_precision(geoms, grid_size=0.00001)
    bairros = bairros[["codbairro", "nome", "regiao_adm", "rp", "geometry"]]

    # Uma resposta incompleta do servidor não pode virar o cache.
    if len(pop) < 150 or len(bairros) < 150:
        raise RuntimeError("download incompleto")
    pop.sort_values("codbairro").to_csv(DATA / "populacao_bairros.csv", index=False)
    bairros.sort_values("codbairro").to_file(DATA / "bairros.geojson", driver="GeoJSON")


DATA.mkdir(parents=True, exist_ok=True)
if all((DATA / name).exists() for name in FILES):
    print("usando os arquivos que já estão em", DATA)
else:
    try:
        for name in FILES:
            urllib.request.urlretrieve(CLASS_REPO + name, DATA / name)
        print("baixados os dois arquivos do IPP do repositório da aula para", DATA)
    except Exception as error:
        print("repositório da aula inacessível:", error)
        download_from_datario()
        print("baixados os dois arquivos do DATA.RIO para", DATA)

## Parte 1: aquecimento com pandas

Comece revisando as quatro operações do pandas que você já conhece: ler, filtrar, contar e juntar. Leia a base, fique só com os registros da janela a partir de `WINDOW_START`, conte-os por `main_reason` e por `neighborhood`, e junte `populacao_bairros.csv` pelo nome. Que nome não encontra população na tabela, e quantos registros ele tem?

**FAÇA 1.1** Leia `data/fogo_cruzado_rio.csv` em `fc_all` (converta a coluna `date` em data) e veja o formato da tabela.

In [ ]:
# fc_all = ____
# fc_all.shape
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
fc_all = pd.read_csv(
    "data/fogo_cruzado_rio.csv", parse_dates=["date"])
fc_all.shape

**FAÇA 1.2** Fique só com os registros da janela (`date >= WINDOW_START`), em `fc`. Quantos registros? Quantos com `police_action`?

In [ ]:
# fc = ____
# com_policia = ____
# len(fc), len(com_policia)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
fc = fc_all[fc_all["date"] >= WINDOW_START]
com_policia = fc[fc["police_action"]]
len(fc), len(com_policia)

**FAÇA 1.3** Conte os registros por `main_reason` com `groupby`. Que fração é “Não identificado”?

In [ ]:
# por_motivo = ____
# por_motivo.sort_values(ascending=False)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
por_motivo = fc.groupby("main_reason").size()
por_motivo.sort_values(ascending=False)
por_motivo["Não identificado"] / len(fc)

**FAÇA 1.4** Conte os registros por `neighborhood`, escreva os nomes da tabela de população do mesmo jeito (com `norm`) e junte a população (`how="left"`).

In [ ]:
# pop = pd.read_csv("data/rio/populacao_bairros.csv")
# pop["neighborhood"] = ____  # os nomes do jeito do Fogo Cruzado
# nomes = ____  # registros por neighborhood: um DataFrame com a coluna n_registros
# nomes = ____  # junte pop por "neighborhood"
# nomes.head()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
pop = pd.read_csv("data/rio/populacao_bairros.csv")
pop["neighborhood"] = pop["nome"].map(norm)
nomes = fc.groupby("neighborhood").size()
nomes = nomes.reset_index(name="n_registros")
nomes = nomes.merge(pop, on="neighborhood", how="left")
nomes.head()

**FAÇA 1.5** O `how="left"` mantém todos os nomes do Fogo Cruzado. Qual deles ficou sem população, e quantos registros ele tem?

In [ ]:
# nomes[____]
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
nomes[nomes["populacao_2022"].isna()]

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# Resposta: `NI`, com 1 registro na janela; a tabela não diz o que a sigla significa. Todos os outros nomes acharam uma linha, o que faz do nome uma chave que parece confiável. A Parte 3 confere o nome contra o lugar.

**FAÇA 1.6** *Extra: a base inteira.* Conte os registros por ano, com e sem vítimas, em `fc_all`, e desenhe em barras empilhadas. 2016 começa em julho e 2025 termina em 24 de novembro.

In [ ]:
# com_vitima = ____
# por_ano = ____
# por_ano
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
com_vitima = fc_all["n_victims"] > 0
por_ano = fc_all.groupby(["year", com_vitima]).size().unstack()
por_ano
ax = por_ano.plot.bar(stacked=True)
ax.legend(["sem vítimas", "com vítimas"])
ax.set_ylabel("registros por ano")

## Parte 2: GeoDataFrame

Crie `pontos` a partir de `fc` com `points_from_xy` e `crs=4326`, leia `bairros.geojson` e desenhe os bairros com os registros por cima. Depois conte quantos pontos distintos há, e quantos registros tem o ponto mais cheio, na janela e na base inteira.

Um GeoDataFrame é o mesmo DataFrame com mais uma coluna, `geometry`. O x é a longitude.

**FAÇA 2.1** Transforme `fc` num GeoDataFrame `pontos` (longitude primeiro, `crs=4326`).

In [ ]:
# xy = ____
# pontos = ____
# pontos.head()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
xy = gpd.points_from_xy(fc.longitude, fc.latitude)
pontos = gpd.GeoDataFrame(fc, geometry=xy, crs=4326)
pontos.head()

**FAÇA 2.2** Leia `data/rio/bairros.geojson` em `bairros`.

In [ ]:
# bairros = ____
# bairros.head()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
bairros = gpd.read_file("data/rio/bairros.geojson")
bairros.head()

Tudo o que vem do pandas continua funcionando num GeoDataFrame:

In [ ]:
com_policia = pontos[pontos["police_action"]]
pontos.groupby("main_reason").size()

**FAÇA 2.3** Desenhe os bairros e, por cima deles, os registros.

In [ ]:
# ax = ____
# pontos.plot(ax=ax, markersize=2)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
ax = bairros.plot(color="tan", edgecolor="grey")
pontos.plot(ax=ax, color="black", markersize=4, alpha=0.5)

**FAÇA 2.4** *Extra.* O mesmo filtro da Parte 1, agora em `pontos`: desenhe só os registros com ação policial.

In [ ]:
# ax = bairros.plot(color="tan", edgecolor="grey")
# ____.plot(ax=ax, color="black", markersize=4, alpha=0.5)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
ax = bairros.plot(color="tan", edgecolor="grey")
com_policia.plot(ax=ax, color="black", markersize=4, alpha=0.5)

**FAÇA 2.5** Quantos pontos distintos há na janela, e quantos registros tem o ponto mais cheio? (Agrupe pelas duas colunas de coordenadas.)

In [ ]:
# por_ponto = ____
# len(por_ponto), por_ponto.max()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
por_ponto = pontos.groupby(["latitude", "longitude"]).size()
len(por_ponto), por_ponto.max()

**FAÇA 2.6** E na base inteira (`fc_all`)? A conta é a mesma, sem precisar de geometria.

In [ ]:
# por_ponto_todos = ____
# len(por_ponto_todos), por_ponto_todos.max()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
por_ponto_todos = fc_all.groupby(["latitude", "longitude"]).size()
len(por_ponto_todos), por_ponto_todos.max()

2.7 Por que isso importa para um mapa de pontos?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# 2.7 Por que isso importa para um mapa de pontos?
#
# Resposta: na janela, 5.129 registros estão em apenas 2.802 pontos distintos, e o ponto mais cheio tem 99 registros; na base inteira, 27.738 registros ficam em 12.473 pontos, e o mais cheio tem 416. Pontos desenhados uns sobre os outros escondem quantos registros cada um representa: o agrupamento de marcadores e o mapa de calor da Parte 6 mostram esse empilhamento.

## Parte 3: junção espacial

Faça o `sjoin` de `pontos` em `bairros` com `predicate="within"`, conte os registros por bairro, junte as contagens a `bairros` e desenhe. Que registro some da junção? Quantos registros têm um nome diferente do polígono, e qual é o maior par, na janela e na base inteira (`fc_all`)?

O `sjoin` é um `merge` com o lugar como chave: cada registro recebe as colunas do bairro em que cai.

**FAÇA 3.1** Junte `pontos` a `bairros` com `predicate="within"`.

In [ ]:
# com_bairro = ____
# com_bairro[["neighborhood", "nome", "regiao_adm"]].head()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
com_bairro = gpd.sjoin(pontos, bairros, predicate="within")
com_bairro[["neighborhood", "nome", "regiao_adm"]].head()

**FAÇA 3.2** Conte os registros por bairro (`codbairro`) e junte as contagens de volta a `bairros`. Use `how="left"` para manter os bairros sem registro.

In [ ]:
# n = ____  # registros por codbairro
# n = n.reset_index(name="n_registros")
# bairros = ____  # junção à esquerda em bairros
# bairros["n_registros"] = bairros["n_registros"].fillna(0)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
n = com_bairro.groupby("codbairro").size()
n = n.reset_index(name="n_registros")
bairros = bairros.merge(n, on="codbairro", how="left")
bairros["n_registros"] = bairros["n_registros"].fillna(0)

**FAÇA 3.3** Desenhe os bairros coloridos por `n_registros`, em cinco faixas de tamanho parecido (`pd.qcut`), com legenda.

In [ ]:
# classe = ____
# bairros.plot(column=____, cmap="Greens", legend=True)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
classe = pd.qcut(bairros["n_registros"], 5)
bairros.plot(column=classe, cmap="Greens", legend=True)

**FAÇA 3.4** Que bairros têm mais registros?

In [ ]:
# bairros.nlargest(3, ____)[["nome", "n_registros"]]
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
bairros.nlargest(3, "n_registros")[["nome", "n_registros"]]

**FAÇA 3.5** O `sjoin` só guarda os registros que caem dentro de um bairro. Que registro falta em `com_bairro`?

In [ ]:
# pontos[____][["date", "neighborhood"]]
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
pontos[~pontos.index.isin(com_bairro.index)][["date", "neighborhood"]]

Sua resposta: (a que distância ele fica?)

In [ ]:
# @title Solução { display-mode: "form" }
# Resposta: um registro chamado GALEAO (17/07/2024) fica a cerca de 43 m do polígono mais próximo, então o `within` não acha bairro para ele. O `sjoin` é uma junção interna por padrão: a linha some sem aviso. Confira o total de registros depois de toda junção (5.129 registros, 5.128 juntados).

**FAÇA 3.6** Compare o nome que o Fogo Cruzado escreveu com o polígono em que o ponto cai. Escreva o nome do polígono do jeito do Fogo Cruzado, com `norm`, e conte os registros em que os dois diferem.

In [ ]:
# nome_ipp = ____  # com_bairro["nome"] do jeito do Fogo Cruzado
# dif = com_bairro[____]  # os registros em que ele difere do nome
# len(dif) / len(com_bairro)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
nome_ipp = com_bairro["nome"].map(norm)
dif = com_bairro[nome_ipp != com_bairro["neighborhood"]]
len(dif) / len(com_bairro)

**FAÇA 3.7** Quais são os maiores pares nome-polígono?

In [ ]:
# dif.groupby([____, ____]).size().nlargest(5)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
dif.groupby(["neighborhood", "nome"]).size().nlargest(5)

3.8 Que coluna você usaria para contar por bairro, e por quê?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# 3.8 Que coluna você usaria para contar por bairro, e por quê?
#
# Uma resposta: a localização. Contar pelo polígono faz a contagem e a população do Censo 2022 usarem os mesmos limites de bairro, e a junção não depende de como um nome foi escrito. Guarde o nome como conferência: onde ele discorda (10,1% dos registros da janela), olhe o par, como CAMPINHO em Cascadura e VILA KENNEDY em Bangu. Nenhuma das colunas é a verdade sobre o evento: um ponto pode ser uma rua ou uma localidade geocodificada, não uma posição de GPS. Pelo nome, Bangu tem 170 registros na janela; pelo polígono, 211: diga qual usou.

**FAÇA 3.9** A mesma conferência na base inteira (`fc_all`, 27.738 registros): que fração discorda, e quais são os maiores pares?

In [ ]:
# todos = ____  # fc_all como pontos, juntado a bairros
# dif_todos = ____
# len(dif_todos) / len(todos)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
todos = gpd.GeoDataFrame(
    fc_all, geometry=gpd.points_from_xy(fc_all.longitude, fc_all.latitude), crs=4326)
todos = gpd.sjoin(todos, bairros[["nome", "geometry"]], predicate="within")
dif_todos = todos[todos["nome"].map(norm) != todos["neighborhood"]]
print(dif_todos.groupby(["neighborhood", "nome"]).size().nlargest(3))
len(dif_todos) / len(todos)

3.10 Qual é o maior par nome-polígono na base inteira?

Sua resposta:

In [ ]:
# @title Solução { display-mode: "form" }
# 3.10 Qual é o maior par nome-polígono na base inteira?
#
# Resposta: VILA KENNEDY dentro de Bangu, com 336 registros. Depois vêm SAO CRISTOVAO, que o IPP escreve “Imperial de São Cristóvão” (91, o mesmo lugar escrito de outro jeito), e COPACABANA dentro de Ipanema (67). Na base inteira, 9,1% dos registros têm um nome diferente do polígono (10,1% na janela).

## Parte 4: contagens e taxas

Junte `populacao_bairros.csv` por `codbairro` e calcule os registros por 100 mil habitantes. Deixe de fora os bairros com menos de 5 mil habitantes, ordene por contagem e por taxa e liste os cinco bairros que mais mudam de posição. Depois escreva o título de um gráfico do seu ranking de modo que ele diga com o que está comparando.

Sozinha, uma contagem não diz nada sobre risco; dividir por habitantes é um primeiro passo, com limites. A janela tem 2,9 anos, então estas não são taxas por ano.

**FAÇA 4.1** Junte a população a `bairros` por `codbairro` (não pelo nome) e calcule os registros por 100 mil habitantes em `por_100k`.

In [ ]:
# pop = pd.read_csv("data/rio/populacao_bairros.csv")
# pop = pop[["codbairro", "populacao_2022"]]
# bairros = ____  # junção à esquerda por codbairro
# taxa = bairros["n_registros"] / bairros["populacao_2022"]
# bairros["por_100k"] = ____
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
pop = pd.read_csv("data/rio/populacao_bairros.csv")
pop = pop[["codbairro", "populacao_2022"]]
bairros = bairros.merge(pop, on="codbairro", how="left")
taxa = bairros["n_registros"] / bairros["populacao_2022"]
bairros["por_100k"] = taxa * 100_000

**FAÇA 4.2** Apague a taxa dos bairros com menos de 5 mil habitantes, ou sem linha de população. Quantos são?

In [ ]:
# pequeno = ____  # True onde populacao_2022 é menor que 5000 ou está faltando
# bairros.loc[pequeno, "por_100k"] = None
# pequeno.sum()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
pequeno = ~(bairros["populacao_2022"] >= 5000)
bairros.loc[pequeno, "por_100k"] = None
pequeno.sum()

**FAÇA 4.3** Ordene os bairros por contagem e por taxa (posição 1 é a maior) e mostre os cinco primeiros por contagem com as duas posições.

In [ ]:
# rk = dict(ascending=False, method="min")
# bairros["rank_n"] = ____
# bairros["rank_taxa"] = ____
# cols = ["nome", "n_registros", "por_100k"]
# cols += ["rank_n", "rank_taxa"]
# bairros.nsmallest(5, "rank_n")[cols]
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
rk = dict(ascending=False, method="min")
bairros["rank_n"] = bairros["n_registros"].rank(**rk)
bairros["rank_taxa"] = bairros["por_100k"].rank(**rk)
cols = ["nome", "n_registros", "por_100k"]
cols += ["rank_n", "rank_taxa"]
bairros.nsmallest(5, "rank_n")[cols]

**FAÇA 4.4** Quanto os dois rankings se parecem? Calcule a correlação de Spearman entre a contagem e a taxa e compare as duas listas dos 10 primeiros.

In [ ]:
# ok = ____  # os bairros que entram nos dois rankings
# rho = ____  # correlação de Pearson das duas colunas de posições (Spearman)
# dez_n = set(bairros.nsmallest(10, "rank_n")["nome"])
# dez_taxa = set(bairros.nsmallest(10, "rank_taxa")["nome"])
# rho, len(dez_n & dez_taxa)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
ok = bairros.dropna(subset=["por_100k"])  # os 154 bairros ranqueados dos dois jeitos
rho = ok["n_registros"].rank().corr(ok["por_100k"].rank())  # Spearman: Pearson nas posições
rho
dez_n = set(bairros.nsmallest(10, "rank_n")["nome"])
dez_taxa = set(bairros.nsmallest(10, "rank_taxa")["nome"])
rho, len(dez_n & dez_taxa)

Sua resposta: (o quanto eles concordam?)

In [ ]:
# @title Solução { display-mode: "form" }
# Resposta: a correlação de posições é cerca de 0,75 nos 154 bairros ranqueados, e as duas listas dos 10 primeiros têm só 5 bairros em comum. Cascadura é o 1º nos dois; Bangu é o 3º por contagem e o 40º por taxa, Jacarepaguá o 9º e o 65º, Campinho o 23º e o 2º. Parecidos, não iguais.

**FAÇA 4.5** Quais cinco bairros mais mudam de posição da contagem para a taxa? (A diferença, em posições, sem o sinal.)

In [ ]:
# mudou = ____  # |rank_n - rank_taxa| em ok
# ok.loc[mudou.nlargest(5).index, ["nome", "rank_n", "rank_taxa"]]
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
mudou = (ok["rank_n"] - ok["rank_taxa"]).abs()
ok.loc[mudou.nlargest(5).index, ["nome", "rank_n", "rank_taxa"]]

Sua resposta: (que bairros são, e o que eles têm em comum?)

In [ ]:
# @title Solução { display-mode: "form" }
# Resposta: Campo Grande (29º por contagem, 135º por taxa), Santa Cruz (18º e 111º), Guaratiba (41º e 119º), Barra da Tijuca (37º e 113º) e Copacabana (51º e 123º): todos caem. Todos têm muitos habitantes: dividida por muita gente, a taxa fica baixa. Ela não diz por que os registros são esses. Campinho, o 2º por taxa, sobe 21 posições.

**FAÇA 4.6** Divida a taxa em cinco faixas de tamanho parecido (`pd.qcut`) e guarde os seis limites (o coropleto da Parte 6 usa os mesmos).

In [ ]:
# classe_taxa, cortes = ____  # retbins=True devolve os limites
# classe_taxa.cat.categories
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
classe_taxa, cortes = pd.qcut(bairros["por_100k"], 5, retbins=True)
classe_taxa.cat.categories

**FAÇA 4.7** Desenhe a contagem e a taxa lado a lado, cada uma em cinco faixas.

In [ ]:
# import matplotlib.pyplot as plt
#
# classe_n = ____
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# bairros.plot(column=____, legend=True, ax=axes[0])
# bairros.plot(column=____, legend=True, ax=axes[1],
#              missing_kwds={"color": "lightgrey"})
# axes[0].set_title("____")
# axes[1].set_title("____")
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
import matplotlib.pyplot as plt

classe_n = pd.qcut(bairros["n_registros"], 5)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bairros.plot(column=classe_n, legend=True, ax=axes[0])
bairros.plot(column=classe_taxa, legend=True, ax=axes[1],
             missing_kwds={"color": "lightgrey"})
axes[0].set_title("Registros por bairro")
axes[1].set_title("Registros por 100 mil habitantes")

4.8 Escreva o título de um gráfico do seu ranking de modo que ele diga com o que está comparando.

Seu título:

In [ ]:
# @title Solução { display-mode: "form" }
# 4.8 Escreva o título de um gráfico do seu ranking de modo que ele diga com o que está comparando.
#
# Um título: "Bangu tem o 3º maior número de registros em 2023-25, mas é o 40º por 100 mil habitantes". Ele diz a conclusão e com o que cada número é comparado. Não diz o porquê.

**FAÇA 4.9** *Extra: um registro.* Vítimas por bairro na janela. Qual bairro lidera e quanto do total dele é um registro só?

In [ ]:
# vitimas = ____  # n_victims somado por "nome"
# vitimas.nlargest(3), com_bairro["n_victims"].nlargest(3)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
vitimas = com_bairro.groupby("nome")["n_victims"].sum()
vitimas.nlargest(3), com_bairro["n_victims"].nlargest(3)

**FAÇA 4.10** *Extra: a base inteira.* O mesmo ranking de contagem contra taxa em `fc_all` (2016-2025). Em que posição fica Bangu?

In [ ]:
# todos = ____  # fc_all como pontos, juntado a bairros
# # contagem por bairro, taxa por 100 mil, ordene dos dois jeitos, como em 4.1 a 4.3
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
todos = gpd.GeoDataFrame(
    fc_all, geometry=gpd.points_from_xy(fc_all.longitude, fc_all.latitude), crs=4326)
todos = gpd.sjoin(todos, bairros[["codbairro", "geometry"]], predicate="within")
n_todos = todos.groupby("codbairro").size().rename("n_todos")
b2 = bairros.merge(n_todos, on="codbairro", how="left")
b2["taxa_todos"] = b2["n_todos"] / b2["populacao_2022"] * 100_000
b2.loc[~(b2["populacao_2022"] >= 5000), "taxa_todos"] = None
b2["rank_n"] = b2["n_todos"].rank(**rk)
b2["rank_taxa"] = b2["taxa_todos"].rank(**rk)
b2[b2["nome"] == "Bangu"][["nome", "n_todos", "rank_n", "rank_taxa"]]

## Parte 5: formas

Escolha outro bairro ou um ano. Calcule a envoltória convexa, o círculo mínimo e o centroide dos registros e desenhe os três sobre os pontos. Que forma você usaria para dizer “a área que estes registros cobrem”, e comparada a quê?

Cada forma é uma chamada de biblioteca: não é preciso saber o algoritmo.

**FAÇA 5.1** Selecione os seus pontos (a aula usou os registros dentro de Bangu) e funda-os numa forma só com `union_all()`.

In [ ]:
# sel = ____  # um bairro, ou um ano
# pts = ____  # todos os pontos como uma forma só
# len(sel)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
sel = com_bairro[com_bairro["nome"] == "Bangu"]
pts = sel.union_all()
len(sel)

**FAÇA 5.2** A envoltória convexa, o círculo mínimo e o centroide de `pts`.

In [ ]:
# hull = ____
# circle = ____
# centro = ____
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
hull = pts.convex_hull
pts_m = sel.to_crs(31983).union_all()  # UTM 23S, em metros
circ_m = shapely.minimum_bounding_circle(pts_m)
circle = gpd.GeoSeries([circ_m], crs=31983).to_crs(4326)[0]
centro = pts.centroid

**FAÇA 5.3** Desenhe a envoltória e o círculo sobre os bairros e os seus pontos, com zoom no círculo.

In [ ]:
# ax = bairros.plot(color="white", edgecolor="grey")
# shapes = ____  # uma GeoSeries com a envoltória e o círculo, crs=4326
# shapes.plot(ax=ax, facecolor="none", edgecolor=["green", "darkgoldenrod"])
# sel.plot(ax=ax, markersize=4)
# ax.plot(centro.x, centro.y, "k+", markersize=15)
# xmin, ymin, xmax, ymax = circle.buffer(0.005).bounds
# ax.set_xlim(xmin, xmax)
# ax.set_ylim(ymin, ymax)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
ax = bairros.plot(color="white", edgecolor="grey")
shapes = gpd.GeoSeries([hull, circle], crs=4326)
shapes.plot(ax=ax, facecolor="none", edgecolor=["green", "darkgoldenrod"])
sel.plot(ax=ax, markersize=4)
ax.plot(centro.x, centro.y, "k+", markersize=15)
xmin, ymin, xmax, ymax = circle.buffer(0.005).bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_title("Envoltória (verde) e círculo (amarelo)")

5.4 Que forma você usaria para dizer “a área que estes registros cobrem”, e comparada a quê?

Sua resposta:

In [ ]:
# @title Solução { display-mode: "form" }
# 5.4 Que forma você usaria para dizer “a área que estes registros cobrem”, e comparada a quê?
#
# Uma resposta: a envoltória, porque ela acompanha os pontos: o círculo abrange bairros vizinhos onde não há registro. Em Bangu a envoltória tem 21 km², o círculo 36 km² e o polígono do próprio Bangu 36 km². Nenhuma é “a” área: o número depende da forma escolhida, e um número precisa de algo para ser comparado, então diga que forma usou e com o que comparou.

**FAÇA 5.5** *Extra.* Áreas em km². Latitude e longitude são graus, não metros, então use os pontos já reprojetados para um CRS métrico (`pts_m` e `circ_m` da 5.2: o EPSG:31983 é o fuso UTM 23S, em metros). Quanto medem a envoltória e o círculo, e quanto mede o polígono do próprio Bangu?

In [ ]:
# hull_km2 = ____
# circle_km2 = ____
# round(hull_km2, 1), round(circle_km2, 1)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
hull_km2 = pts_m.convex_hull.area / 1e6
circle_km2 = circ_m.area / 1e6
round(hull_km2, 1), round(circle_km2, 1)
bairro_km2 = bairros.to_crs(31983).area[bairros["nome"] == "Bangu"].iloc[0] / 1e6
round(bairro_km2, 1)

## Parte 6: o mapa

Num mapa folium: marcadores com balões para um subconjunto que você escolher, todos os registros num MarkerCluster, um HeatMap, e a envoltória, o círculo e o centroide da parte 5; depois salve com `m.save("map.html")` e abra o arquivo no navegador. Extra: um coropleto de registros por 100 mil habitantes e um LayerControl.

O folium quer `[lat, lon]`, que é `[y, x]`.

Um mapa salvo guarda os registros que mostra: compartilhe como compartilharia os registros.

**FAÇA 6.1** Um mapa centrado em `[-22.91, -43.40]` com zoom 11.

In [ ]:
# m = ____
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
m = folium.Map(location=[-22.91, -43.40], zoom_start=11)

**FAÇA 6.2** Um marcador com balão por registro de um subconjunto pequeno (a aula usou os 47 registros com 5 ou mais vítimas).

In [ ]:
# grandes = ____  # um subconjunto pequeno, por exemplo fc["n_victims"] >= 5
# for _, row in grandes.iterrows():
#     texto = f"{row.neighborhood}: {row.n_victims} vítimas"
#     folium.Marker(
#         ____,  # [lat, lon]
#         popup=texto,
#     ).add_to(m)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
grandes = fc[fc["n_victims"] >= 5]
for _, row in grandes.iterrows():
    texto = f"{row.neighborhood}: {row.n_victims} vítimas"
    folium.Marker(
        [row.latitude, row.longitude], popup=texto,
    ).add_to(m)

**FAÇA 6.3** Todos os registros, adicionados a um `MarkerCluster` em vez de ao mapa.

In [ ]:
# cluster = ____
# cluster.add_to(m)
# for _, row in fc.iterrows():
#     folium.Marker(
#         [row.latitude, row.longitude],
#         popup=row.neighborhood,
#     ).add_to(cluster)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
cluster = plugins.MarkerCluster(name="registros")
cluster.add_to(m)
for _, row in fc.iterrows():
    folium.Marker(
        [row.latitude, row.longitude],
        popup=row.neighborhood,
    ).add_to(cluster)

**FAÇA 6.4** Um `HeatMap` a partir de uma lista de pares `[lat, lon]`.

In [ ]:
# coords = ____
# plugins.HeatMap(coords, name="calor", radius=12).add_to(m)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
coords = fc[["latitude", "longitude"]].values.tolist()
plugins.HeatMap(coords, name="calor", radius=12).add_to(m)

6.5 O mapa de calor mostra onde os registros foram feitos. É onde os tiroteios acontecem, onde o Fogo Cruzado ficou sabendo deles, ou onde as pessoas moram? Por quanto você dividiria para comparar dois bairros?

Sua resposta:

In [ ]:
# @title Solução { display-mode: "form" }
# 6.5 O mapa de calor mostra onde os registros foram feitos. É onde os tiroteios acontecem, onde o Fogo Cruzado ficou sabendo deles, ou onde as pessoas moram? Por quanto você dividiria para comparar dois bairros?
#
# Uma resposta: a tabela só conhece o que foi registrado, então o mapa de calor mostra onde os registros foram feitos, e um ponto que aparece duas vezes conta duas vezes. Ele não mostra onde os eventos acontecem, nem quantas pessoas moram ali. Para comparar dois bairros, divida por alguma coisa: os habitantes (Parte 4), e diga o que esse denominador deixa de fora.

**FAÇA 6.6** A sua envoltória (verde) e o seu círculo (amarelo) da parte 5 com `folium.GeoJson` (o 2º argumento é o `style_function`, que devolve a cor), cada uma com o seu `name=`, e o centroide como um marcador.

In [ ]:
# verde = lambda _: {"color": "green"}
# amarelo = lambda _: {"color": "darkgoldenrod"}
# folium.GeoJson(____, verde, name="envoltória").add_to(m)
# folium.GeoJson(____, amarelo, name="círculo").add_to(m)
# folium.Marker(____).add_to(m)  # o centroide, como [y, x]
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
verde = lambda _: {"color": "green"}
amarelo = lambda _: {"color": "darkgoldenrod"}
folium.GeoJson(hull, verde, name="envoltória").add_to(m)
folium.GeoJson(circle, amarelo, name="círculo").add_to(m)
folium.Marker([centro.y, centro.x]).add_to(m)

**FAÇA 6.7** *Extra* (precisa do 4.2 e do 4.6). Um coropleto de registros por 100 mil habitantes, nas mesmas cinco faixas, e depois um `LayerControl` para ligar e desligar as camadas.

In [ ]:
# folium.Choropleth(
#     geo_data=bairros,
#     data=bairros,
#     columns=["codbairro", "por_100k"],
#     key_on=____,  # onde o codbairro fica em cada feição do GeoJSON
#     fill_color="YlGn",
#     bins=____,  # os limites das cinco faixas, de 4.6
#     nan_fill_color="grey",
#     nan_fill_opacity=0.6,
#     legend_name="registros por 100 mil habitantes",
# ).add_to(m)
# folium.LayerControl().add_to(m)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
folium.Choropleth(
    geo_data=bairros,
    data=bairros,
    columns=["codbairro", "por_100k"],
    key_on="feature.properties.codbairro",
    fill_color="YlGn",
    bins=list(cortes),
    nan_fill_color="grey",
    nan_fill_opacity=0.6,
    legend_name="registros por 100 mil habitantes",
).add_to(m)
folium.LayerControl().add_to(m)

**FAÇA 6.8** *Extra.* Densidade sem biblioteca de mapas: um hexbin dos registros. Mude o `gridsize`: o que muda?

In [ ]:
# ax = bairros.plot(color="white", edgecolor="grey")
# ax.hexbin(____, ____, gridsize=40, mincnt=1)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
ax = bairros.plot(color="white", edgecolor="grey")
ax.hexbin(pontos.geometry.x, pontos.geometry.y,
          gridsize=40, mincnt=1, bins="log")

**FAÇA 6.9** Salve o mapa como `map.html` e mostre aqui.

In [ ]:
# m.save(____)
# m
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
m.save("map.html")
m

`map.html` é um arquivo só: abre em qualquer navegador, sem Python. É o mapa que você pode mandar para alguém. Ele leva junto os registros que mostra.

## Parte 7 (extra): comece o seu trabalho

Use o tempo livre para o que você vai entregar: escolha um gráfico ou um mapa de hoje, dê a ele um título que diga o que ele compara e anote o que os dados não mostram sobre ele. Um gráfico basta.